<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/01b_near_duplicate_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Near-duplicate check on the current splits

Manuscript section 3.5 (leakage check). Exact duplicates were removed before splitting, but the same artwork can also appear
re-saved, resized or recompressed, which a file hash misses. If such a copy lands in training and another in test,
the test images aren't truly unseen and the CNN results are optimistic.

This compares every image against every image in the other splits using a 64-bit perceptual hash (dHash) and lists pairs that differ
in at most 5 of the 64 bits. **It only reads the prepared data and changes nothing**: the splits, models and results stay as they are.
The report goes to Google Drive (`thesis-data/data_quality/`).

**No GPU needed**: Runtime > Change runtime type > **CPU**. A few minutes, mostly unzipping.

## 1. Get the code

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd /content/repo
!pip install -q -e .

## 2. Restore the prepared data from Google Drive (read only)

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
D = "/content/drive/MyDrive/thesis-data"
ZIP = f"{D}/harmonized.zip"
if not os.path.exists(ZIP):
    raise FileNotFoundError(f"{ZIP} not found: run notebook 01 through step 8 first")
!unzip -qo $ZIP -d .
!mkdir -p $D/data_quality && rm -rf outputs/data_quality && ln -s $D/data_quality outputs/data_quality
!wc -l data/harmonized/train.csv data/harmonized/val.csv data/harmonized/test.csv

## 3. Run the check

In [ ]:
!python scripts/check_near_duplicates.py

## 4. Look at the pairs
Hash distance 0 means the two thumbnails match exactly; 5 is the loosest match counted. The first grid shows the closest pairs,
the second the loosest ones (distance 4-5), which is where unrelated but similar-looking images (flat colours, plain backgrounds)
would show up if the threshold were too loose. Note in each pair whether it is really the same artwork.

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
summary = json.load(open("outputs/data_quality/near_duplicates_summary.json"))
print(json.dumps({k: v for k, v in summary.items() if k != "splits_dir"}, indent=1))
pairs = pd.read_csv("outputs/data_quality/near_duplicates.csv")
LABEL = {0: "human", 1: "AI"}

def show(rows, title):
    if rows.empty:
        print(f"{title}: none"); return
    fig, axes = plt.subplots(len(rows), 2, figsize=(6, 3 * len(rows)), squeeze=False)
    for ax_row, (_, r) in zip(axes, rows.iterrows()):
        for ax, side in zip(ax_row, "ab"):
            with Image.open(r[f"path_{side}"]) as im:
                ax.imshow(im.convert("RGB"))
            ax.set_title(f"{r[f'split_{side}']} | {LABEL[r[f'label_{side}']]} | {r[f'source_{side}']}", fontsize=8)
            ax.axis("off")
        ax_row[0].text(-0.05, 0.5, f"d={r.hash_distance}", transform=ax_row[0].transAxes, ha="right", va="center")
    fig.suptitle(title); plt.tight_layout(); plt.show()

show(pairs.nsmallest(8, "hash_distance"), "Closest pairs")
show(pairs[pairs.hash_distance >= 4].sample(min(8, (pairs.hash_distance >= 4).sum()), random_state=0), "Loosest pairs (distance 4-5)")
conflicts = pairs[pairs.label_a != pairs.label_b]
show(conflicts.head(8), "Pairs with conflicting labels (same image labelled human and AI)")

## 5. What to do with the result
Save this notebook to GitHub (File > Save a copy in GitHub) so the numbers can be reviewed. Roughly:

- **No pairs, or only a handful that aren't real duplicates:** the current splits are fine; keep all existing results.
- **Real duplicates across splits:** rebuild the splits with one copy of each removed, then retrain and rerun notebooks 02-05.
  How many is "too many" is a judgement call for the group: compare the share of test images involved with the size of the effects being reported.
- **Conflicting labels:** the same image is labelled human in one dataset and AI in another; those should be dropped whatever the count.